In [1]:
!pip install -q openpyxl tqdm rapidfuzz

from openpyxl import load_workbook
from tqdm.auto import tqdm
from rapidfuzz import fuzz
import re
import html
import unicodedata

# =========================
# 1. File paths
# =========================

MAIN_FILE = "/content/export5_tmp_lsf_full_dryrun_after_doc_type_fix.xlsx"
CORRECTED_LCNAF_FILE = "/content/LCNAF_updated.xlsx"
CORRECTED_GEO_FILE = "/content/GeoNames_updated.xlsx"

OUTPUT_FILE = "/content/export6_tmp_lsf_full_dryrun_after_doc_type_fix.xlsx"

HEADER_ROW = 1

SUBJECT_PERSON_COL = "subject_person_600"
SUBJECT_GEO_COL = "subject_geo_651"

CREATOR_LINKED_NEW_COL = "creator_linked_new"
RECEIVER_LINKED_NEW_COL = "receiver_linked_new"

SENDER_PLACE_LINKED_NEW_COL = "sender_place_linked_new"
RECEIVER_PLACE_LINKED_NEW_COL = "receiver_place_linked_new"

UNCONTROLLED_PERSON_COL = "sub_uncontrolled_person"
UNCONTROLLED_PLACE_COL = "sub_uncontrolled_place"

# corrected_lcnaf columns
LC_URI_COL = "LODLOC_uri"
LC_LABEL_COL = "LODLOC_label"
LC_PERSON_COL = "personStdName"

# corrected_geo columns
GEO_URI_COL_CANDIDATES = [
    "LODgeoName_Updated",
    "LODgeoName_updated",
    "LODgeoName_uri",
    "LODgeoName_URI"
]

GEO_LABEL_COL_CANDIDATES = [
    "geoLabel_final2_final",
    "geoLabel_final",
    "geoLabel_final2",
    "geoLabel",
    "GeoNames_label"
]

FUZZY_PLACE_THRESHOLD = 90
ONE_WORD_PLACE_THRESHOLD = 97
AMBIGUITY_SCORE_GAP = 2


# =========================
# 2. Basic helpers
# =========================

def get_header_positions(ws):
    positions = {}

    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value

        if value is not None:
            header = str(value).strip()
            positions.setdefault(header, []).append(col)

    return positions


def get_first_col(headers, col_name, required=True):
    if col_name in headers:
        return headers[col_name][0]

    if required:
        raise ValueError(f"Missing required column: {col_name}")

    return None


def get_first_available_col(headers, candidates, required=True):
    for col_name in candidates:
        if col_name in headers:
            return col_name, headers[col_name][0]

    if required:
        raise ValueError(f"Missing all candidate columns: {candidates}")

    return None, None


def get_or_create_col(ws, headers, col_name):
    if col_name in headers:
        return headers[col_name][0]

    new_idx = ws.max_column + 1
    ws.cell(HEADER_ROW, new_idx).value = col_name
    headers[col_name] = [new_idx]

    print(f"Created column: {col_name}")

    return new_idx


def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_space(text):
    if text is None:
        return ""

    return re.sub(r"\s+", " ", str(text)).strip()


def strip_tags(text):
    if text is None:
        return ""

    text = re.sub(r"<[^>]+>", " ", str(text))
    text = html.unescape(text)

    return normalize_space(text)


def normalize_text_for_match(text):
    if text is None:
        return ""

    text = strip_tags(text)
    text = text.strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def split_subject_values(value):
    """
    Split subject values on strong separators only.
    Do not split on comma because names/places contain commas.
    """
    if is_blank(value):
        return []

    text = normalize_space(value)

    parts = [
        p.strip()
        for p in re.split(r"\s*;\s*|\s*\|\s*|\n+", text)
        if p.strip()
    ]

    return parts


# =========================
# 3. LCNAF helpers
# =========================

def extract_lc_id(value):
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(r"[a-z]{1,4}\d+[a-z0-9]*", text, flags=re.IGNORECASE):
        return text.replace(".html", "").strip()

    m = re.search(
        r"id\.loc\.gov/authorities/names/([A-Za-z0-9]+)(?:\.html)?",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    return None


def normalize_lc_uri(uri):
    loc_id = extract_lc_id(uri)

    if loc_id:
        return f"http://id.loc.gov/authorities/names/{loc_id}"

    if uri is None:
        return ""

    return (
        str(uri)
        .strip()
        .replace("https://id.loc.gov", "http://id.loc.gov")
        .replace(".html", "")
        .rstrip("/")
    )


def build_lc_anchor(uri, label):
    uri = html.escape(normalize_lc_uri(uri), quote=True)
    label = html.escape(str(label).strip(), quote=False)

    return f'<p><a href="{uri}">{label}</a> [LC]</p>'


def strip_life_dates_for_match(name):
    if name is None:
        return ""

    name = strip_tags(name)

    name = re.sub(
        r"\s*\([^)]*(?:\d{3,4}|b\.|d\.|fl\.|ca\.|c\.)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\s*,\s*-?\d{3,4}\s*(?:[-–]\s*\d{0,4})?",
        "",
        name
    )

    name = re.sub(r"\b\d{3,4}\s*[-–]\s*\d{0,4}\b", "", name)
    name = re.sub(r"\b\d{3,4}\b", "", name)

    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s+,", ",", name).strip()
    name = re.sub(r",\s*$", "", name).strip()

    return name


def person_std_to_full_name(person_std):
    person_std = strip_life_dates_for_match(person_std)

    if "," not in person_std:
        return person_std

    last, first = person_std.split(",", 1)

    last = normalize_space(last)
    first = normalize_space(first)

    if first and last:
        return f"{first} {last}"

    return person_std


def full_name_to_person_std(name):
    """
    Roger Haydock -> Haydock, Roger
    James Harrison -> Harrison, James
    """
    name = strip_life_dates_for_match(name)

    if "," in name:
        return name

    parts = name.split()

    if len(parts) >= 2:
        last = parts[-1]
        first = " ".join(parts[:-1])

        return f"{last}, {first}"

    return name


def person_lookup_keys(name):
    keys = set()

    clean = strip_life_dates_for_match(name)

    if not clean:
        return keys

    keys.add(normalize_text_for_match(clean))

    if "," in clean:
        keys.add(normalize_text_for_match(person_std_to_full_name(clean)))
    else:
        keys.add(normalize_text_for_match(full_name_to_person_std(clean)))

    return {k for k in keys if k}


# =========================
# 4. GeoNames helpers
# =========================

def extract_geo_id(value):
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(r"\d+", text):
        return text

    m = re.search(
        r"geonames\.org/(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    return None


def normalize_geo_uri(uri):
    geoid = extract_geo_id(uri)

    if geoid:
        return f"https://sws.geonames.org/{geoid}"

    if uri is None:
        return ""

    return str(uri).strip().rstrip("/")


def build_geo_anchor(uri, label):
    uri = html.escape(normalize_geo_uri(uri), quote=True)
    label = html.escape(str(label).strip(), quote=False)

    return f'<p><a href="{uri}">{label}</a> [GeoNames]</p>'


def base_place_name(label):
    if label is None:
        return ""

    text = strip_tags(label)
    text = re.sub(r"\s*\([^)]*\)\s*$", "", text).strip()

    return text


# =========================
# NEW AMBIGUOUS GEONAMES RESOLVER
# =========================

def before_first_bracket(text):
    """
    For corrected GeoNames label:
    'Glaucha (Halle (Saale), Kreisfreie Stadt Halle, ...)' -> 'Glaucha'
    """
    if text is None:
        return ""

    text = strip_tags(text)
    text = text.split("(", 1)[0].strip()

    return text


def before_first_comma(text):
    """
    For subject geo/place value:
    'Glaucha, Halle (Saale), Kreisfreie Stadt Halle, ...' -> 'Glaucha'
    """
    if text is None:
        return ""

    text = strip_tags(text)
    text = text.split(",", 1)[0].strip()

    return text


def normalized_prefix_match(main_place, corrected_label):
    """
    Extra ambiguity condition:
    If label before first bracket in corrected geoLabel_final2
    matches label before first comma in the subject geo/place name,
    then use that GeoNames record.
    """
    main_prefix = normalize_place_for_match(before_first_comma(main_place))
    corrected_prefix = normalize_place_for_match(before_first_bracket(corrected_label))

    if not main_prefix or not corrected_prefix:
        return False

    return main_prefix == corrected_prefix


# =========================
# End new ambiguous GeoNames resolver
# =========================


def normalize_place_for_match(text):
    if text is None:
        return ""

    text = strip_tags(text)
    text = text.strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.lower()

    text = re.sub(r"\bschloss\b", "castle", text)
    text = re.sub(r"\bburg\b", "castle", text)

    text = re.sub(
        r"\b(city of|town of|village of|municipality of|county of|district of)\b",
        " ",
        text
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)

    text = re.sub(
        r"\b(city|town|village|municipality|county|district|province|state|region|country|place)\b",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def place_match_score(main_place, corrected_label):
    main_norm = normalize_place_for_match(main_place)

    corrected_full = normalize_place_for_match(corrected_label)
    corrected_base = normalize_place_for_match(base_place_name(corrected_label))

    if not main_norm or not corrected_base:
        return 0

    if main_norm == corrected_base or main_norm == corrected_full:
        return 100

    main_words = main_norm.split()
    base_words = corrected_base.split()

    if len(main_words) >= 2 and len(base_words) >= 2:
        if corrected_base in main_norm or main_norm in corrected_base:
            return 96

    return max(
        fuzz.token_set_ratio(main_norm, corrected_base),
        fuzz.token_set_ratio(main_norm, corrected_full),
        fuzz.token_sort_ratio(main_norm, corrected_base),
        fuzz.token_sort_ratio(main_norm, corrected_full)
    )


def is_safe_place_score(main_place, corrected_label, score):
    main_norm = normalize_place_for_match(main_place)
    corrected_base = normalize_place_for_match(base_place_name(corrected_label))

    main_words = main_norm.split()
    corrected_words = corrected_base.split()

    if len(main_words) <= 1 or len(corrected_words) <= 1:
        return score >= ONE_WORD_PLACE_THRESHOLD

    return score >= FUZZY_PLACE_THRESHOLD


# =========================
# 5. Load corrected_lcnaf
# =========================

lc_wb = load_workbook(CORRECTED_LCNAF_FILE, data_only=True)

lc_ws = None
lc_headers = None

for sheet in lc_wb.worksheets:
    temp_headers = get_header_positions(sheet)

    if LC_URI_COL in temp_headers and LC_LABEL_COL in temp_headers:
        lc_ws = sheet
        lc_headers = temp_headers
        break

if lc_ws is None:
    raise ValueError("Could not find corrected_lcnaf sheet with LODLOC_uri and LODLOC_label.")

print("Using corrected LCNAF sheet:", lc_ws.title)

lc_uri_idx = get_first_col(lc_headers, LC_URI_COL)
lc_label_idx = get_first_col(lc_headers, LC_LABEL_COL)
lc_person_idx = get_first_col(lc_headers, LC_PERSON_COL, required=False)

lc_lookup = {}
lc_records = []

for row in range(HEADER_ROW + 1, lc_ws.max_row + 1):
    uri = lc_ws.cell(row, lc_uri_idx).value
    label = lc_ws.cell(row, lc_label_idx).value
    person = lc_ws.cell(row, lc_person_idx).value if lc_person_idx else None

    if is_blank(uri) or is_blank(label):
        continue

    loc_id = extract_lc_id(uri)

    if not loc_id:
        continue

    rec = {
        "source": "corrected_lcnaf",
        "row": row,
        "id": loc_id,
        "uri": normalize_lc_uri(uri),
        "label": str(label).strip(),
        "anchor": build_lc_anchor(uri, label)
    }

    lc_records.append(rec)

    for name in [person, label]:
        for key in person_lookup_keys(name):
            lc_lookup.setdefault(key, []).append(rec)

print("LCNAF corrected records:", len(lc_records))
print("LCNAF lookup keys:", len(lc_lookup))


# =========================
# 6. Load corrected_geo
# =========================

geo_wb = load_workbook(CORRECTED_GEO_FILE, data_only=True)

geo_ws = None
geo_headers = None

for sheet in geo_wb.worksheets:
    temp_headers = get_header_positions(sheet)

    has_uri = any(c in temp_headers for c in GEO_URI_COL_CANDIDATES)
    has_label = any(c in temp_headers for c in GEO_LABEL_COL_CANDIDATES)

    if has_uri and has_label:
        geo_ws = sheet
        geo_headers = temp_headers
        break

if geo_ws is None:
    raise ValueError("Could not find corrected_geo sheet with GeoNames URI and label columns.")

print("Using corrected GeoNames sheet:", geo_ws.title)

geo_uri_col_used, geo_uri_idx = get_first_available_col(geo_headers, GEO_URI_COL_CANDIDATES)
geo_label_col_used, geo_label_idx = get_first_available_col(geo_headers, GEO_LABEL_COL_CANDIDATES)

print("Using GeoNames URI column:", geo_uri_col_used)
print("Using GeoNames label column:", geo_label_col_used)

geo_records = []

for row in range(HEADER_ROW + 1, geo_ws.max_row + 1):
    uri = geo_ws.cell(row, geo_uri_idx).value
    label = geo_ws.cell(row, geo_label_idx).value

    if is_blank(uri) or is_blank(label):
        continue

    geoid = extract_geo_id(uri)

    if not geoid:
        continue

    rec = {
        "source": "corrected_geo",
        "row": row,
        "id": geoid,
        "uri": normalize_geo_uri(uri),
        "label": str(label).strip(),
        "anchor": build_geo_anchor(uri, label)
    }

    geo_records.append(rec)

print("GeoNames corrected records:", len(geo_records))


# =========================
# 7. Load main workbook
# =========================

wb = load_workbook(MAIN_FILE)

ws = None
headers = None

for sheet in wb.worksheets:
    temp_headers = get_header_positions(sheet)

    required_cols = [
        SUBJECT_PERSON_COL,
        SUBJECT_GEO_COL,
        CREATOR_LINKED_NEW_COL,
        RECEIVER_LINKED_NEW_COL,
        SENDER_PLACE_LINKED_NEW_COL,
        RECEIVER_PLACE_LINKED_NEW_COL
    ]

    if all(col in temp_headers for col in required_cols):
        ws = sheet
        headers = temp_headers
        break

if ws is None:
    raise ValueError("Could not find main sheet with subject and linked columns.")

print("Using main sheet:", ws.title)

subject_person_idx = get_first_col(headers, SUBJECT_PERSON_COL)
subject_geo_idx = get_first_col(headers, SUBJECT_GEO_COL)

creator_linked_idx = get_first_col(headers, CREATOR_LINKED_NEW_COL)
receiver_linked_idx = get_first_col(headers, RECEIVER_LINKED_NEW_COL)

sender_place_linked_idx = get_first_col(headers, SENDER_PLACE_LINKED_NEW_COL)
receiver_place_linked_idx = get_first_col(headers, RECEIVER_PLACE_LINKED_NEW_COL)

uncontrolled_person_idx = get_or_create_col(ws, headers, UNCONTROLLED_PERSON_COL)
uncontrolled_place_idx = get_or_create_col(ws, headers, UNCONTROLLED_PLACE_COL)


# =========================
# 8. Parse anchors from existing linked columns
# =========================

ANY_ANCHOR_RE = re.compile(
    r'(?P<full><a\b[^>]*\bhref=["\'](?P<href>[^"\']+)["\'][^>]*>(?P<label>.*?)</a>)',
    re.IGNORECASE | re.DOTALL
)


def extract_lc_anchors_from_cell(cell_value):
    results = []

    if is_blank(cell_value):
        return results

    text = str(cell_value)

    for m in ANY_ANCHOR_RE.finditer(text):
        href = m.group("href")
        label = strip_tags(m.group("label"))

        loc_id = extract_lc_id(href)

        if not loc_id:
            continue

        uri = normalize_lc_uri(href)

        results.append({
            "source": "main_linked_column",
            "id": loc_id,
            "uri": uri,
            "label": label,
            "anchor": build_lc_anchor(uri, label)
        })

    return results


def extract_geo_anchors_from_cell(cell_value):
    results = []

    if is_blank(cell_value):
        return results

    text = str(cell_value)

    for m in ANY_ANCHOR_RE.finditer(text):
        href = m.group("href")
        label = strip_tags(m.group("label"))

        geoid = extract_geo_id(href)

        if not geoid:
            continue

        uri = normalize_geo_uri(href)

        results.append({
            "source": "main_linked_column",
            "id": geoid,
            "uri": uri,
            "label": label,
            "anchor": build_geo_anchor(uri, label)
        })

    return results


def unique_records_by_id(records):
    unique = {}

    for rec in records:
        unique[rec["id"]] = rec

    return list(unique.values())


def choose_unique_person_match(records):
    records = unique_records_by_id(records)

    if len(records) == 1:
        return records[0], "matched"

    if len(records) > 1:
        return None, "ambiguous"

    return None, "not_found"


def find_person_from_corrected(name):
    candidates = []

    for key in person_lookup_keys(name):
        candidates.extend(lc_lookup.get(key, []))

    return choose_unique_person_match(candidates)


def find_person_from_row_fallback(name, row):
    row_records = []

    for col_idx in [creator_linked_idx, receiver_linked_idx]:
        row_records.extend(
            extract_lc_anchors_from_cell(ws.cell(row, col_idx).value)
        )

    candidates = []
    target_keys = person_lookup_keys(name)

    for rec in row_records:
        rec_keys = person_lookup_keys(rec["label"])

        if target_keys.intersection(rec_keys):
            candidates.append(rec)

    return choose_unique_person_match(candidates)


def choose_unique_geo_match(records_scores, place_name=None):
    if not records_scores:
        return None, "not_found", 0

    records_scores.sort(key=lambda x: x[0], reverse=True)

    best_score = records_scores[0][0]

    close = [
        (score, rec)
        for score, rec in records_scores
        if best_score - score <= AMBIGUITY_SCORE_GAP
    ]

    unique = {}

    for score, rec in close:
        unique[rec["id"]] = (score, rec)

    if len(unique) == 1:
        score, rec = list(unique.values())[0]
        return rec, "matched", score

    # =========================
    # NEW AMBIGUOUS GEONAMES RESOLVER
    # =========================
    # If multiple GeoNames records are close/ambiguous,
    # select the single record where:
    # subject place before first comma
    # matches corrected GeoNames label before first bracket.

    if place_name is not None and len(unique) > 1:
        prefix_matches = []

        for score, rec in unique.values():
            if normalized_prefix_match(place_name, rec["label"]):
                prefix_matches.append((score, rec))

        unique_prefix = {}

        for score, rec in prefix_matches:
            unique_prefix[rec["id"]] = (score, rec)

        if len(unique_prefix) == 1:
            score, rec = list(unique_prefix.values())[0]
            return rec, "matched_prefix_resolved", score

    # =========================
    # End new ambiguous GeoNames resolver
    # =========================

    return None, "ambiguous", best_score


def find_geo_from_corrected(place_name):
    scores = []

    for rec in geo_records:
        score = place_match_score(place_name, rec["label"])

        if is_safe_place_score(place_name, rec["label"], score):
            scores.append((score, rec))

    return choose_unique_geo_match(scores, place_name)


def find_geo_from_row_fallback(place_name, row):
    row_records = []

    for col_idx in [sender_place_linked_idx, receiver_place_linked_idx]:
        row_records.extend(
            extract_geo_anchors_from_cell(ws.cell(row, col_idx).value)
        )

    scores = []

    for rec in row_records:
        score = place_match_score(place_name, rec["label"])

        if is_safe_place_score(place_name, rec["label"], score):
            scores.append((score, rec))

    return choose_unique_geo_match(scores, place_name)


# =========================
# 9. Update subject_person_600
# =========================

person_log = []
place_log = []

person_ambig_log = []
place_ambig_log = []

person_updated_count = 0
place_updated_count = 0

for row in tqdm(
    range(HEADER_ROW + 1, ws.max_row + 1),
    desc="Updating subject_person_600"
):
    original_value = ws.cell(row, subject_person_idx).value

    if is_blank(original_value):
        ws.cell(row, subject_person_idx).value = None
        ws.cell(row, uncontrolled_person_idx).value = None
        continue

    names = split_subject_values(original_value)

    new_anchors = []
    uncontrolled = []

    for name in names:
        rec, status = find_person_from_corrected(name)
        source_used = "corrected_lcnaf"

        if rec is None:
            if status == "ambiguous":
                person_ambig_log.append([
                    row,
                    name,
                    "corrected_lcnaf",
                    "ambiguous corrected_lcnaf match"
                ])

            rec, status = find_person_from_row_fallback(name, row)
            source_used = "main creator/receiver linked columns"

        if rec is not None:
            new_anchors.append(rec["anchor"])

            person_log.append([
                row,
                name,
                source_used,
                rec["id"],
                rec["uri"],
                rec["label"]
            ])

        else:
            uncontrolled.append(name)

            person_log.append([
                row,
                name,
                "uncontrolled",
                "",
                "",
                ""
            ])

    # IMPORTANT:
    # subject_person_600 should contain ONLY links.
    # If nothing matched, clear the old names from subject_person_600.
    ws.cell(row, subject_person_idx).value = "".join(new_anchors) if new_anchors else None

    # Unmatched names go here.
    ws.cell(row, uncontrolled_person_idx).value = "; ".join(uncontrolled) if uncontrolled else None

    if new_anchors:
        person_updated_count += 1


print("Rows with LCNAF links in subject_person_600:", person_updated_count)


# =========================
# 10. Update subject_geo_651
# =========================

for row in tqdm(
    range(HEADER_ROW + 1, ws.max_row + 1),
    desc="Updating subject_geo_651"
):
    original_value = ws.cell(row, subject_geo_idx).value

    if is_blank(original_value):
        ws.cell(row, subject_geo_idx).value = None
        ws.cell(row, uncontrolled_place_idx).value = None
        continue

    places = split_subject_values(original_value)

    new_anchors = []
    uncontrolled = []

    for place in places:
        rec, status, score = find_geo_from_corrected(place)
        source_used = "corrected_geo"

        if rec is None:
            if status == "ambiguous":
                place_ambig_log.append([
                    row,
                    place,
                    "corrected_geo",
                    "ambiguous corrected_geo match",
                    score
                ])

            rec, status, score = find_geo_from_row_fallback(place, row)
            source_used = "main sender/receiver place linked columns"

        if rec is not None:
            new_anchors.append(rec["anchor"])

            place_log.append([
                row,
                place,
                source_used,
                rec["id"],
                rec["uri"],
                rec["label"],
                score
            ])

        else:
            uncontrolled.append(place)

            place_log.append([
                row,
                place,
                "uncontrolled",
                "",
                "",
                "",
                score
            ])

    # IMPORTANT:
    # subject_geo_651 should contain ONLY GeoNames links.
    # If nothing matched, clear the old places from subject_geo_651.
    ws.cell(row, subject_geo_idx).value = "".join(new_anchors) if new_anchors else None

    # Unmatched places go here.
    ws.cell(row, uncontrolled_place_idx).value = "; ".join(uncontrolled) if uncontrolled else None

    if new_anchors:
        place_updated_count += 1


print("Rows with GeoNames links in subject_geo_651:", place_updated_count)


# =========================
# 11. Add log sheets
# =========================

log_sheet_names = [
    "sub_person_log",
    "sub_place_log",
    "sub_person_ambig",
    "sub_place_ambig"
]

for sheet_name in log_sheet_names:
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]


ws_person_log = wb.create_sheet("sub_person_log")
ws_person_log.append([
    "export_row",
    "subject_person_name",
    "source_used",
    "lcnaf_id",
    "lcnaf_uri",
    "lcnaf_label"
])

for item in person_log:
    ws_person_log.append(item)


ws_place_log = wb.create_sheet("sub_place_log")
ws_place_log.append([
    "export_row",
    "subject_place_name",
    "source_used",
    "geonames_id",
    "geonames_uri",
    "geonames_label",
    "match_score"
])

for item in place_log:
    ws_place_log.append(item)


ws_person_ambig = wb.create_sheet("sub_person_ambig")
ws_person_ambig.append([
    "export_row",
    "subject_person_name",
    "source_checked",
    "reason"
])

for item in person_ambig_log:
    ws_person_ambig.append(item)


ws_place_ambig = wb.create_sheet("sub_place_ambig")
ws_place_ambig.append([
    "export_row",
    "subject_place_name",
    "source_checked",
    "reason",
    "score"
])

for item in place_ambig_log:
    ws_place_ambig.append(item)


for ws_log in [
    ws_person_log,
    ws_place_log,
    ws_person_ambig,
    ws_place_ambig
]:
    for col in range(1, ws_log.max_column + 1):
        ws_log.column_dimensions[ws_log.cell(1, col).column_letter].width = 34


# =========================
# 12. Save safely
# =========================

for sheet in wb.worksheets:
    sheet.sheet_state = "visible"

wb.active = wb.sheetnames.index(ws.title)

if wb.views:
    wb.views[0].activeTab = wb.sheetnames.index(ws.title)
    wb.views[0].firstSheet = 0

wb.save(OUTPUT_FILE)

print("Done.")
print("Saved file:", OUTPUT_FILE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.3 MB/s eta 0:00:00
Using corrected LCNAF sheet: LCNAFlabel_check
LCNAF corrected records: 229
LCNAF lookup keys: 593
Using corrected GeoNames sheet: PRINT place full GeoNamesupdate
Using GeoNames URI column: LODgeoName_Updated
Using GeoNames label column: geoLabel_final
GeoNames corrected records: 462
Using main sheet: Sheet1
Created column: sub_uncontrolled_person
Created column: sub_uncontrolled_place


Updating subject_person_600:   0%|          | 0/478 [00:00<?, ?it/s]

Rows with LCNAF links in subject_person_600: 0


Updating subject_geo_651:   0%|          | 0/478 [00:00<?, ?it/s]

Rows with GeoNames links in subject_geo_651: 165
Done.
Saved file: /content/export6_tmp_lsf_full_dryrun_after_doc_type_fix.xlsx
